In [2]:
%uv pip install scikit-learn ipywidgets jupyterlab

Using Python 3.12.6 environment at: /usr/local
Resolved 100 packages in 356ms
⠙ Preparing packages... (0/36)
⠙ Preparing packages... (0/36)
⠙ Preparing packages... (0/36)
fqdn       ------------------------------ 8.91 KiB/8.91 KiB
⠙ Preparing packages... (0/36)
fqdn       ------------------------------ 8.91 KiB/8.91 KiB
⠙ Preparing packages... (0/36)
fqdn       ------------------------------ 8.91 KiB/8.91 KiB
isoduration ------------------------------     0 B/11.06 KiB
⠙ Preparing packages... (0/36)
fqdn       ------------------------------ 8.91 KiB/8.91 KiB
isoduration ------------------------------ 11.06 KiB/11.06 KiB
⠙ Preparing packages... (0/36)
fqdn       ------------------------------ 8.91 KiB/8.91 KiB
uri-template ------------------------------     0 B/10.88 KiB
isoduration ------------------------------ 11.06 KiB/11.06 KiB
⠙ Preparing packages... (0/36)
fqdn       ------------------------------ 8.91 KiB/8.91 KiB
uri-template ------------------------------ 10.88 KiB/10.88 KiB
i

In [1]:
import math

import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 32, max_len: int = 1500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        
        return x

class IOHTransformer(nn.Module):
    def __init__(self, model_dim=64, num_heads=8):
        super(IOHTransformer, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=model_dim, num_heads=num_heads, batch_first=True)
        self.layer_norm_attn = nn.LayerNorm(model_dim)

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.ReLU(),
            nn.Linear(model_dim * 4, model_dim)
        )
        self.layer_norm_ffn = nn.LayerNorm(model_dim)

    def forward(self, x):
        # x shape: (Batch, seq_len, input_dim)
        attn_output, _ = self.multihead_attn(x, x, x)  # (Batch, seq_len, model_dim)
        x = self.layer_norm_attn(x + attn_output)  # Residual connection + LayerNorm

        ffn_output = self.ffn(x)  # (Batch, seq_len, model_dim)
        x = self.layer_norm_ffn(x + ffn_output)  # Residual connection + LayerNorm

        return x

class IOHPredictor(nn.Module):
    def __init__(self, input_dim = 4, model_dim_1 = 32, model_dim_2 = 64, num_heads = 4):
        super(IOHPredictor, self).__init__()
        self.input_projection_1 = nn.Linear(input_dim, model_dim_1)
        self.pos_embed = PositionalEncoding(d_model=model_dim_1)
        self.transformer_1 = IOHTransformer(model_dim=model_dim_1, num_heads=num_heads)

        self.conv1d = nn.Conv1d(model_dim_1, model_dim_2, kernel_size=5, stride=2)

        # self.input_projection_2 = nn.Linear(model_dim_1, model_dim_2)
        self.transformer_2 = IOHTransformer(model_dim=model_dim_2, num_heads=num_heads)
        self.output_projection = nn.Sequential(
            nn.Linear(model_dim_2+5, model_dim_2 // 2),
            nn.GELU(),
            nn.Linear(model_dim_2 // 2, 1)
        )
    
    def forward(self, x_seq, x_static):
        x = self.input_projection_1(x_seq) # (B, 900, 4)
        x = self.pos_embed(x)  # (B, 900, 4) -> (B, 900, model_dim)
        x = self.transformer_1(x)  # (Batch, 900, 32)

        # x = self.input_projection_2(x)
        x = self.conv1d(x.transpose(1, 2)).transpose(1, 2)
        x = torch.nn.functional.gelu(x)
        x = self.transformer_2(x)  # (Batch, 900, 64)

        x = torch.mean(x, dim=1)  # (B, 64)
        x = torch.concat([x, x_static], dim=-1)  # (B, 64 + 5)

        output = self.output_projection(x)  # (B, 1)
        return output.squeeze(-1)  # (B,)

if __name__ == "__main__":
    model = IOHPredictor(input_dim=4, model_dim_1=32, model_dim_2=64, num_heads=4)
    dummy_seq = torch.randn(2, 900, 4)  # (Batch, seq_len, input_dim)
    dummy_static = torch.randn(2, 5)# (Batch, static_dim)

    print("Parameters: ", sum(p.numel() for p in model.parameters()))

    output = model(dummy_seq, dummy_static)
    print("Output shape:", output.shape)

Parameters:  75425
Output shape: torch.Size([2])


In [2]:
import os
from typing import Dict
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

class IOHDataset(Dataset):
    def __init__(self, data_dir: str, manifest: Dict[int, int]):
        super().__init__()
        self.data_dir = data_dir
        
        # 1. Calculate total windows to pre-allocate exact memory
        total_windows = sum(manifest.values())
        print(f"Pre-allocating RAM for {total_windows} windows...")

        # 2. Pre-allocate massive empty tensors (Zero memory spike!)
        self.X_seq = torch.empty((total_windows, 900, 4), dtype=torch.float32)
        self.X_static = torch.empty((total_windows, 5), dtype=torch.float32)
        self.Y = torch.empty((total_windows,), dtype=torch.long)

        # 3. Fill the tensors directly from disk
        current_idx = 0
        for case_id in tqdm(sorted(manifest.keys())):
            n_windows = manifest[case_id]
            if n_windows == 0:
                continue
                
            path = os.path.join(data_dir, f"case_{case_id}.pt")
            data = torch.load(path, weights_only=True)
            assert data["X_seq"].shape[0] == n_windows, \
                f"case_{case_id}: manifest says {n_windows} windows but file has {data['X_seq'].shape[0]}"
            
            # Slot the data into the pre-allocated block
            end_idx = current_idx + n_windows
            self.X_seq[current_idx:end_idx] = data["X_seq"]
            self.X_static[current_idx:end_idx] = data["X_static"]
            self.Y[current_idx:end_idx] = data["Y"]
            
            current_idx = end_idx

        print(f"Dataset successfully loaded into RAM. Size: {self.X_seq.element_size() * self.X_seq.nelement() / 1e9:.2f} GB")

    def __len__(self) -> int:
        return len(self.Y)

    def __getitem__(self, idx: int):
        return self.X_seq[idx], self.X_static[idx], self.Y[idx]

In [ ]:
import os
import random
import pickle
import time  # <-- Added for timing
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np
import functools
from tqdm.auto import tqdm

TRAIN_DIR = "/mnt/dataforioh/processed_data/train"
TEST_DIR = "/mnt/dataforioh/processed_data/test"
OUTPUT_DIR = "/mnt/dataforioh/processed_data"

# 1. CONFIGURATION & HYPERPARAMETERS
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 50
PATIENCE = 5  # Early stopping patience
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

def set_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

def main():
    print(f"========== Starting Training on {DEVICE} ==========")

    # 2. LOAD METADATA & PATIENT-LEVEL SPLIT
    meta_path = os.path.join(OUTPUT_DIR, "pipeline_meta.pkl")
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)

    full_train_manifest = meta["manifest"]["train"]
    test_manifest = meta["manifest"]["test"]

    # Extract patient IDs (keys) from the training manifest
    train_case_ids = list(full_train_manifest.keys())

    # Patient-Level Validation Split (80% Train, 20% Val)
    actual_train_ids, val_ids = train_test_split(train_case_ids, test_size=0.2, random_state=42)

    # Rebuild the manifests for Train and Val
    train_manifest = {cid: full_train_manifest[cid] for cid in actual_train_ids}
    val_manifest = {cid: full_train_manifest[cid] for cid in val_ids}

    print(f"Patients -> Train: {len(actual_train_ids)} | Val: {len(val_ids)} | Test: {len(test_manifest.keys())}")

    # 3. BUILD DATALOADERS
    train_ds = IOHDataset(TRAIN_DIR, train_manifest)
    val_ds = IOHDataset(TRAIN_DIR, val_manifest)  # Val uses TRAIN_DIR because it was split from train
    test_ds = IOHDataset(TEST_DIR, test_manifest)

    # Pin memory speeds up CPU-to-GPU data transfers
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    
    # 4. MODEL, LOSS, & OPTIMIZER SETUP
    model = IOHPredictor(input_dim=4, model_dim_1=32, model_dim_2=64, num_heads=4)
    model.to(DEVICE)

    # BCEWithLogitsLoss is required for binary classification with raw logit outputs
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    # 5. TRAINING & VALIDATION LOOP (WITH EARLY STOPPING)
    best_val_auprc = 0.0
    epochs_no_improve = 0

    min_delta = 0.001

    for epoch in range(EPOCHS):
        # --- START TIMING & RESET VRAM ---
        epoch_start_time = time.time()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats(DEVICE)

        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        
        for X_seq, X_static, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Training"):
            X_seq, X_static = X_seq.to(DEVICE), X_static.to(DEVICE)
            # CRITICAL: Cast int64 labels to float32 for BCEWithLogitsLoss
            labels = labels.float().to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_seq, X_static)
            
            loss = criterion(logits, labels)
            loss.backward()
            
            # CRITICAL FIX: Gradient clipping to prevent Mamba NaN errors
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item() * X_seq.size(0)
            
        train_loss /= len(train_loader.dataset)

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for X_seq, X_static, labels in val_loader:
                X_seq, X_static = X_seq.to(DEVICE), X_static.to(DEVICE)
                labels = labels.float().to(DEVICE)

                logits = model(X_seq, X_static)
                loss = criterion(logits, labels)
                val_loss += loss.item() * X_seq.size(0)

                # Convert logits to probabilities using Sigmoid for metrics
                probs = torch.sigmoid(logits)
                all_val_preds.extend(probs.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        
        # Calculate Metrics
        val_auprc = average_precision_score(all_val_labels, all_val_preds)
        val_auroc = roc_auc_score(all_val_labels, all_val_preds)

        scheduler.step(val_auprc)

        # --- END TIMING & GRAB PEAK VRAM ---
        epoch_duration = time.time() - epoch_start_time
        if DEVICE.type == "cuda":
            peak_vram_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 * 1024)
        else:
            peak_vram_mb = 0.0

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Time: {epoch_duration:.2f}s | Peak VRAM: {peak_vram_mb:.1f} MB")
        print(f"             | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUPRC: {val_auprc:.4f} | Val AUROC: {val_auroc:.4f}")

        # --- EARLY STOPPING CHECK ---
        if val_auprc > (best_val_auprc + min_delta):
            best_val_auprc = val_auprc
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_transformer_weights.pth")
            print("  -> Validation AUPRC improved. Model saved.")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"\nEarly stopping triggered after {epoch+1} epochs.")
                break

    # 6. FINAL TESTING PHASE (BLIND VAULT)
    print("\n========== Evaluating on Holdout Test Set ==========")
    # Load the best weights discovered during validation
    model.load_state_dict(torch.load("best_transformer_weights.pth", weights_only=True))
    model.eval()
    
    all_test_preds = []
    all_test_labels = []

    with torch.no_grad():
        for X_seq, X_static, labels in test_loader:
            X_seq, X_static = X_seq.to(DEVICE), X_static.to(DEVICE)
            labels = labels.float().to(DEVICE)

            logits = model(X_seq, X_static)
            probs = torch.sigmoid(logits)
            
            all_test_preds.extend(probs.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())

    test_auprc = average_precision_score(all_test_labels, all_test_preds)
    test_auroc = roc_auc_score(all_test_labels, all_test_preds)

    print(f"FINAL TEST METRICS | AUPRC: {test_auprc:.4f} | AUROC: {test_auroc:.4f}")
    print("====================================================")

if __name__ == "__main__":
    set_seed(42)
    main()

========== Starting Training on cuda ==========
Patients -> Train: 2176 | Val: 545 | Test: 698
Pre-allocating RAM for 75844 windows...


  0%|          | 0/2176 [00:00<?, ?it/s]

Dataset successfully loaded into RAM. Size: 1.09 GB
Pre-allocating RAM for 20172 windows...


  0%|          | 0/545 [00:00<?, ?it/s]

Dataset successfully loaded into RAM. Size: 0.29 GB
Pre-allocating RAM for 92751 windows...


...

Epoch 3/50 Training:   0%|          | 0/2371 [00:00<?, ?it/s]

Epoch [3/50] | Time: 114.64s | Peak VRAM: 1626.6 MB
             | Train Loss: 0.4928 | Val Loss: 0.5045 | Val AUPRC: 0.4840 | Val AUROC: 0.7354
  -> Validation AUPRC improved. Model saved.


Epoch 4/50 Training:   0%|          | 0/2371 [00:00<?, ?it/s]

Epoch [4/50] | Time: 114.66s | Peak VRAM: 1626.6 MB
             | Train Loss: 0.4888 | Val Loss: 0.5133 | Val AUPRC: 0.4843 | Val AUROC: 0.7397


Epoch 5/50 Training:   0%|          | 0/2371 [00:00<?, ?it/s]

In [ ]:
def test_eval(ckpt_path, test_loader):
    # 6. FINAL TESTING PHASE (BLIND VAULT)
    print("\n========== Evaluating on Holdout Test Set ==========")
    # Load the best weights discovered during validation
    model.load_state_dict(torch.load("best_transformer_weights.pth", weights_only=True))
    model.eval()
    
    all_test_preds = []
    all_test_labels = []

    with torch.no_grad():
        for X_seq, X_static, labels in test_loader:
            X_seq, X_static = X_seq.to(DEVICE), X_static.to(DEVICE)
            labels = labels.float().to(DEVICE)

            logits = model(X_seq, X_static)
            probs = torch.sigmoid(logits)
            
            all_test_preds.extend(probs.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())

    all_test_preds  = np.array(all_test_preds)
    all_test_labels = np.array(all_test_labels)

    # ── Point estimates ───────────────────────────────────────────────────────
    test_auprc  = average_precision_score(all_test_labels, all_test_preds)
    test_auroc  = roc_auc_score(all_test_labels, all_test_preds)

    # Accuracy at 0.5 decision threshold
    binary_preds = (all_test_preds >= 0.5).astype(int)
    test_accuracy = (binary_preds == all_test_labels.astype(int)).mean()

    # ── Bootstrap Confidence Intervals (95%, 2000 resamples) ─────────────────
    N_BOOTSTRAP  = 2000
    CI_ALPHA     = 0.05          # 95% CI
    rng          = np.random.default_rng(seed=42)
    n_samples    = len(all_test_labels)

    boot_auroc, boot_auprc, boot_acc = [], [], []

    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n_samples, size=n_samples)   # resample with replacement
        b_labels = all_test_labels[idx]
        b_preds  = all_test_preds[idx]

        # Skip degenerate bootstrap draws that contain only one class
        if len(np.unique(b_labels)) < 2:
            continue

        boot_auroc.append(roc_auc_score(b_labels, b_preds))
        boot_auprc.append(average_precision_score(b_labels, b_preds))
        boot_acc.append(((b_preds >= 0.5).astype(int) == b_labels.astype(int)).mean())

    boot_auroc = np.array(boot_auroc)
    boot_auprc = np.array(boot_auprc)
    boot_acc   = np.array(boot_acc)

    lo, hi = CI_ALPHA / 2, 1 - CI_ALPHA / 2          # 2.5th and 97.5th percentiles

    auroc_ci = (np.percentile(boot_auroc, lo * 100), np.percentile(boot_auroc, hi * 100))
    auprc_ci = (np.percentile(boot_auprc, lo * 100), np.percentile(boot_auprc, hi * 100))
    acc_ci   = (np.percentile(boot_acc,   lo * 100), np.percentile(boot_acc,   hi * 100))

    # ── Print results ─────────────────────────────────────────────────────────
    print(f"\n{'Metric':<12} {'Score':>8}   {'95% CI':>22}")
    print("-" * 46)
    print(f"{'AUROC':<12} {test_auroc:>8.4f}   ({auroc_ci[0]:.4f} – {auroc_ci[1]:.4f})")
    print(f"{'AUPRC':<12} {test_auprc:>8.4f}   ({auprc_ci[0]:.4f} – {auprc_ci[1]:.4f})")
    print(f"{'Accuracy':<12} {test_accuracy:>8.4f}   ({acc_ci[0]:.4f} – {acc_ci[1]:.4f})")
    print("-" * 46)
    print(f"Bootstrap resamples : {len(boot_auroc):,}  (seed=42, threshold=0.50)")
    print("====================================================")

meta_path = os.path.join(OUTPUT_DIR, "pipeline_meta.pkl")
with open(meta_path, "rb") as f:
    meta = pickle.load(f)

test_manifest = meta["manifest"]["test"]
test_ds = IOHDataset(TEST_DIR, test_manifest)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

test_eval('/root/best_transformer_weights.pth', test_loader)